# Step 20.4 — Resume Training

## 从保存的说明与数组，重新建立同一组可继续更新的对象

20.1 保存权重，20.2 保存 Adam 状态树，20.3 保存配置、字符映射、进度和文件引用。本节学习 **resume（恢复后继续）**：按顺序核对这些记录，再把恢复的模型、优化器、训练单步与数据采样连接起来。

重点是对象和状态的配对，不是新增训练算法。主线先读恢复协议与小段连接代码，旧 MiniGPT、优化器 loader、loss/clip 与 sampler 完整定义放在可选复用附录。最后保留**唯一一份注释调用顺序**；以后实际调用前仍须定义全部依赖并提供真实文件、原运行配置、数据与 RNG。本轮没有调用它。

先在仓库根目录同步：

```bash
git pull
```

再建立不覆盖笔记的独立时间戳 work 副本：

```bash
mkdir -p notebooks/work && cp -n notebooks/90_step20_4_resume_training.ipynb "notebooks/work/90_step20_4_resume_training_$(date +%Y%m%d_%H%M%S)_work.ipynb"
```

以下所有代码单元未执行、无输出。只准备材料，不导入课程模块运行、不读取语料、不加载 checkpoint、不执行模型、训练或恢复验证。固定 Python 3.12 / MLX v0.32.2；纸面计数和路径不代表已有产物或实测结果。

## 1. 首先区分三个目标

| 目标 | 至少要做什么？ | 不能据此声称什么？ |
|---|---|---|
| 重新得到模型输出 | 同结构、同 token 映射、正确加载权重 | Adam 历史已恢复 |
| Continue optimization：接着优化 | 再恢复对应 Adam 状态、核对进度与配置、使用合适数据 | 后续每次抽样都与未中断时相同 |
| Exact trajectory reproduction：复现原轨迹 | 还需适用的 RNG、sampler/数据顺序、调度器和相容环境等 | 仅靠初始 seed 或文件 hash 就能保证 |

把权重记作 $\theta_k$，Adam 状态记作 $S_k$，其中 $k$ 是已完成更新次数。恢复目标是同一保存边界的 $(\theta_k,S_k)$，不是“随便一份权重 + 另一份也能加载的 optimizer”。元数据的 `completed_updates` 是对 $k$ 的记录，还需与实际 Adam step 核对。

`stoi`、architecture、artifact hash 和计数互相吻合，有助于发现错误；它们仍不是同一 run 来源的完整证明。若原记录无法确认配对，应停止而不是拼出一组看似能运行的文件。

## 2. 当前有效接口：只接收明确的 metadata 文件路径

旧版笔记曾先定义 `load_checkpoint_metadata(checkpoint_dir)`，随后又用同名函数定义 `load_checkpoint_metadata(metadata_path)`。Python 后定义覆盖前定义，完整顺序定义后真正暴露的是**文件路径版**；两个参数不是可以混用的重载。

本版去掉未被调用的早期目录版，只保留原最终文件路径版。它返回 `(metadata_path, metadata, stoi)`，第一项是解析后的 JSON **文件路径**，不是 checkpoint 根目录。

| 接口 | 输入 | 输出/效果 |
|---|---|---|
| `load_checkpoint_metadata(metadata_path)` | 选定的外层 JSON 文件 | 文件路径、metadata 字典、重建的 stoi |
| `resolve_checkpoint_artifacts(artifact_root, metadata)` | 保存时约定的共同根、上面的字典 | 三个已核查文件路径 |
| `create_model_and_optimizer(metadata)` | 架构记录；另须核实全局 ADAM_CONFIG | 一对新建对象，尚未恢复其历史 |
| `load_model_weights_strict(model, weights_path)` | 该 model 与权重路径 | 加载该 model 并求值 |
| `restore_optimizer_state_from_files(optimizer, arrays_path, manifest_path)` | 该 optimizer 与两个配对文件 | 校验后替换状态树 |
| `verify_completed_update_count(optimizer, metadata)` | 恢复后的 optimizer | 核对并返回实际计数 |

下面保留原导入与配置。`CHECKPOINT_DIR`、`METADATA_NAME`、`WEIGHTS_NAME` 等是旧布局提示常量，**最终调用链不靠它们寻找文件**；不要因为它们存在就把所有产物塞进同一目录。实际路径来自显式选择和 metadata 引用。

`ADAM_CONFIG` 仍是课程候选，不是从 checkpoint 自动发现的配置。20.3 外层 metadata 没保存完整 Adam 超参数，未来必须先与原运行记录核对，不能盲用候选值。

In [ ]:
import hashlib
import json
from pathlib import Path

import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
from mlx.utils import tree_flatten, tree_map, tree_unflatten

# Explicit location only; never auto-select a "latest" checkpoint.
CHECKPOINT_DIR = Path("results/checkpoints/REPLACE_WITH_REAL_RUN_DIRECTORY")
METADATA_NAME = "run_metadata.json"
WEIGHTS_NAME = "model.safetensors"
OPTIMIZER_ARRAYS_NAME = "optimizer_state.npz"
OPTIMIZER_MANIFEST_NAME = "optimizer_state.json"

# Must match the original run; values here are course candidates, not discovered metadata.
ADAM_CONFIG = {
    "learning_rate": 3e-4,
    "betas": [0.9, 0.999],
    "eps": 1e-8,
    "bias_correction": True,  # Explicit course choice; MLX v0.32.2 default is False.
}

### 两个文件辅助函数：定义不等于已经读取

`require_file` 只负责要求文件存在并返回 Path；它不检查 JSON 格式或文件内容。`sha256_file` 将文件以二进制打开，每次读取至多 1 MiB，把各块依次交给同一个摘要对象，最后返回十六进制 SHA-256。

`iter(callable, b"")` 重复调用读取动作，遇到空字节串停止；不是给每个块各算一个独立摘要。它与 20.3 的整个文件字节摘要语义一致，只是分块读取。两函数都仅定义，当前没有文件 IO。

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def require_file(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Required checkpoint artifact is missing: {path.resolve()}")
    return path

## 3. 先重建 tokenizer，再核对 architecture 的词表大小

重用 20.3 的三字符例子：`itos_by_id=["a","b","猫"]` 通过 `enumerate` 重建 `{"a":0,"b":1,"猫":2}`。`reconstruct_stoi` 检查字符表类型、长度、单字符要求、重复字符、保存的 stoi 与按 ID 序列化后的 SHA-256。

如果将 `a/b` 的 ID 交换，仍然是三个字符，模型数组 shape 可能完全一样，但 token 行语义改变。所以不能只检查 `vocab_size=3`。摘要检查也不能修复错误映射，遇到冲突应停止。

下方是与 20.3 相同规范化协议的本节消费者：按 ID 顺序、紧凑 JSON、UTF-8，再计算摘要。它不读语料来重新推断词表，不把新词表适配进旧权重。

In [ ]:
def reconstruct_stoi(tokenizer):
    if tokenizer.get("type") != "character":
        raise ValueError("Unsupported tokenizer type.")
    chars = tokenizer.get("itos_by_id")
    if not isinstance(chars, list) or len(chars) != tokenizer.get("vocab_size"):
        raise ValueError("Invalid saved character table.")
    if any(not isinstance(ch, str) or len(ch) != 1 for ch in chars):
        raise ValueError("Invalid character entry.")
    stoi = {ch: i for i, ch in enumerate(chars)}
    if len(stoi) != len(chars) or tokenizer.get("stoi") != stoi:
        raise ValueError("Tokenizer mapping mismatch.")
    canonical = json.dumps([[ch, i] for i, ch in enumerate(chars)],
                           ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    if hashlib.sha256(canonical).hexdigest() != tokenizer.get("stoi_sha256"):
        raise ValueError("Tokenizer fingerprint mismatch.")
    return stoi

### 外层读取函数真正做了哪些事？

1. `require_file(metadata_path).resolve()` 接受 JSON 文件，不能传一个目录。
2. 读取 JSON，要求外层协议 `format_version == 1`。
3. 要求 architecture 是字典且包含五个必需字段；这里的检查只是字段存在，不是完整架构验证。
4. 重建 tokenizer，检查其长度与 architecture 的 `vocab_size` 一致；返回独立字典副本。

函数内保留的 `root = metadata_path.parent` 没有参与返回值或 artifact 拼接。真正解析产物时还要显式传 `artifact_root`。后面的 MiniGPT 构造器继续检查正维度、头数整除等条件；文件存在和 JSON 字段齐全不代表权重已经兼容。

In [ ]:
def load_checkpoint_metadata(metadata_path):
    metadata_path = require_file(metadata_path).resolve()
    root = metadata_path.parent
    with metadata_path.open("r", encoding="utf-8") as stream:
        metadata = json.load(stream)
    if metadata.get("format_version") != 1:
        raise ValueError("Unsupported metadata format version.")
    architecture = metadata.get("architecture")
    required = ("dims", "num_heads", "num_blocks", "max_seq_len", "vocab_size")
    if not isinstance(architecture, dict) or any(k not in architecture for k in required):
        raise ValueError("Architecture metadata is incomplete.")
    stoi = reconstruct_stoi(metadata.get("tokenizer", {}))
    if len(stoi) != architecture["vocab_size"]:
        raise ValueError("Architecture vocab_size and tokenizer mapping disagree.")
    return metadata_path, metadata, dict(stoi)

## 4. 两条路径与三份引用：沿目录图真正拼一次

沿用 20.3 的假想布局，根目录可以是项目的 `results`：

```text
results/                                            ← artifact_root
├── model_weights/minigpt_weights_20261101T010000Z/
│   └── model.safetensors
├── optimizer_states/adam_state_20261101T010000Z/
│   ├── optimizer_state.npz
│   └── optimizer_state.json
└── run_metadata/minigpt_run_20261101T010001Z/
    └── run_metadata.json                           ← metadata_path
```

这张图不是已经存在的文件。权重的记录相对路径是 `model_weights/minigpt_weights_20261101T010000Z/model.safetensors`。把它接到 artifact_root 后才得到正确位置；若接到 metadata_path.parent，就会错误地在 `run_metadata/minigpt_run_.../model_weights/...` 找权重。

`resolve_artifact` 的顺序是：检查记录形式 → 将相对路径与共同根拼接并 resolve → 确认仍处于根内 → 要求现有文件 → 检查大小 → 检查字节摘要 → 返回 Path。`relative_to(root)` 失败表示路径越界，不应放宽检查。记录的相对布局若随整棵目录搬迁保持不变，可以选择新的显式根，但仍须重新核对字节。

`resolve_checkpoint_artifacts` 固定读取 `weights`、`optimizer_arrays`、`optimizer_manifest` 三个键。缺失、大小/hash 不符均会抛异常，完整调用链不进入对象恢复。它不证明三份文件来自同一保存时刻；文件名或时间戳也不是这种证明。

In [ ]:
def resolve_artifact(artifact_root, record, label):
    if not isinstance(record, dict) or not isinstance(record.get("relative_path"), str):
        raise ValueError(f"Missing artifact metadata for {label}.")
    root = Path(artifact_root).resolve()
    path = (root / record["relative_path"]).resolve()
    try:
        path.relative_to(root)
    except ValueError as exc:
        raise ValueError(f"Artifact path escapes artifact root: {label}") from exc
    require_file(path)
    if path.stat().st_size != record.get("size_bytes"):
        raise ValueError(f"Artifact size mismatch: {label}")
    if sha256_file(path) != record.get("sha256"):
        raise ValueError(f"Artifact SHA-256 mismatch: {label}")
    return path

### 把三份结果作为同一组传下去

下面字典中的路径来自同一个 metadata 和同一个 artifact_root，而不是每个地方各自搜索“最近文件”。整个函数成功返回后，才持有一组通过本层检查的路径。文件应保持稳定；检查之后若被其他程序修改，当前代码没有文件锁或不可变快照保护。

In [ ]:
def resolve_checkpoint_artifacts(artifact_root, metadata):
    # artifact_root is the SAME root passed to 20.3 artifact_metadata(...),
    # not necessarily the directory containing run_metadata.json.
    artifacts = metadata.get("artifacts", {})
    return {
        "weights": resolve_artifact(artifact_root, artifacts.get("weights"), "weights"),
        "optimizer_arrays": resolve_artifact(artifact_root, artifacts.get("optimizer_arrays"), "optimizer_arrays"),
        "optimizer_manifest": resolve_artifact(artifact_root, artifacts.get("optimizer_manifest"), "optimizer_manifest"),
    }

## 5. 新建对象只是模板，加载后才是恢复的参数与状态

`reconstruct_model(metadata)` 把五个 architecture 字段按名字传给原 MiniGPT 构造器；本节的同名类在附录保留完整定义。Python 在**调用函数时**寻找 MiniGPT，因此可以先阅读这里的连接函数，再阅读附录；实际调用前必须完成附录定义。

`load_model_weights_strict` 对明确文件调用 `model.load_weights(..., strict=True)`，再求值参数。严格模式用于发现参数名称/shape 等不匹配，不能把它说成 token 映射、训练来源或所有数值有效性的完整验证。不要为了“能加载”改成不严格模式。

In [ ]:
def reconstruct_model(metadata):
    cfg = metadata["architecture"]
    return MiniGPT(
        vocab_size=cfg["vocab_size"],
        dims=cfg["dims"],
        num_heads=cfg["num_heads"],
        num_blocks=cfg["num_blocks"],
        max_seq_len=cfg["max_seq_len"],
    )

def load_model_weights_strict(model, weights_path):
    require_file(weights_path)
    model.load_weights(str(weights_path), strict=True)
    mx.eval(model.parameters())
    return model

### 为什么还要初始化 Adam，再恢复其状态？

`create_model_and_optimizer` 创建一个新的 model，并用已经核对的 ADAM_CONFIG 创建 optimizer。`optimizer.init(model.trainable_parameters())` 根据这份模型参数树建立目标状态模板；这时的 m/v 与 step 是新状态，不是旧训练历史。

后续先将对应权重加载进**同一个 model**，再把 20.2 保存的状态装入**同一个 optimizer**。原顺序是在加载权重前先 init 模板；因此目标结构与 dtype 必须与原保存对象相容。若后续校验发现 dtype/shape 不匹配，应查原配置与记录，不能删掉检查或把零状态冒充恢复状态。

注意，当前 learning_rate 数组的恢复不是自动恢复学习率调度器；原运行若有 scheduler，规则和进度须另行明确。Adam 的 betas/eps/bias correction 等也不能只靠状态叶子数值推断。

In [ ]:
def create_model_and_optimizer(metadata):
    model = reconstruct_model(metadata)
    optimizer = optim.Adam(**ADAM_CONFIG)
    optimizer.init(model.trainable_parameters())
    return model, optimizer

### 20.2 loader 的两个检查层次与状态替换点

完整 `restore_optimizer_state_from_files` 移到附录 B；它是既有协议的复用，而不是本节新算法。这里抓住“何时可以向前走”：

| 层次 | 实际检查 | 失败的含义 |
|---|---|---|
| 文件自己的说明 | optimizer manifest 版本 2、类名、NPZ 数组 key、shape/dtype、Python 标量类型 | 文件组本身不能按这个协议解释 |
| 对照新 optimizer 模板 | 路径集合、数组/非数组身份、各叶 shape/dtype 或类型 | 不能装进当前目标结构 |
| 构造候选并赋回 | `tree_unflatten` 后 `optimizer.state = rebuilt` | 此处才替换状态树，然后求值 |

例如目标 `weight.m` 为 `(2,3)`、保存值为 `(3,2)`，元素个数相同也不应放行。step 是标量数组，shape `()`，不是长度为 1 的向量。当前检查**不要求 step 的数值与新模板零值相同**，否则永远不能恢复非零训练进度。

赋值前的显式结构检查失败不会执行后面的状态赋值，但整个恢复过程不是事务：此前 model 权重可能已加载，state 赋值后的求值也可能失败，没有回滚。不要捕获异常后直接继续训练；应重新核实状态和文件，必要时从新对象重新走完整链。

## 6. 进度核对：不能拿新 optimizer 的 0 当作已恢复

`find_named_step_leaf` 递归遍历字典、列表/元组，收集键名为 `step` 的位置，要求恰好一处；缺失或多个都报错，不猜哪一个是 Adam step。

`verify_completed_update_count` 检查 metadata 的 `completed_updates` 是非负整数，取得 optimizer 的 step 数组、显式求值，再读取一个标量并比较。这里依赖上一步状态结构校验；函数本身没有单独写完整 shape/dtype 白名单。`item()` 要求单元素，但不能把它等同于完整 scalar schema 验证。

纸面上，metadata 写 7、恢复后的 step 为 7，才通过这个计数核对；如果误用新建零 optimizer，0 与 7 不同，应停止。即使同为 7，也只是计数一致，不能推出两份权重与 optimizer 一定同源。

In [ ]:
def find_named_step_leaf(tree):
    matches = []
    def walk(node, path=()):
        if isinstance(node, dict):
            for key, value in node.items():
                next_path = path + (str(key),)
                if str(key) == "step":
                    matches.append((next_path, value))
                walk(value, next_path)
        elif isinstance(node, (list, tuple)):
            for index, value in enumerate(node):
                walk(value, path + (str(index),))
    walk(tree)
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one Adam step leaf, found {len(matches)}.")
    return matches[0]

### 返回值是什么？

返回的 `actual` 是已核对的更新数，不是“已执行本轮训练”的次数。后续循环应从这个保存边界继续；不要把计数重置为 0，也不要把已计划的总数填进 completed 字段。

In [ ]:
def verify_completed_update_count(optimizer, metadata):
    progress = metadata.get("training_progress", {})
    expected = progress.get("completed_updates")
    if type(expected) is not int or expected < 0:
        raise ValueError("Metadata must contain a real non-negative completed_updates value.")
    path, step_value = find_named_step_leaf(optimizer.state)
    if not isinstance(step_value, mx.array):
        raise TypeError(f"Adam step at {path} is not an MLX array.")
    mx.eval(step_value)
    actual = int(step_value.item())
    if actual != expected:
        raise ValueError(f"Adam step {actual} does not match metadata completed_updates {expected}.")
    return actual

# Call only after restoring the optimizer state in a future work copy.

## 7. 重新绑定 train_step：所有调用必须指向同一对恢复对象

**Closure（闭包）**是函数保留外层作用域对象的一种方式。`make_resumed_train_step(model, optimizer)` 在当前 model 上构造新的 `nn.value_and_grad(model, loss_fn)`，返回的 `stable_step` 会使用传入的这一个 model 和 optimizer。

不要直接复用旧 kernel 的 train_step：它可能仍绑定中断前或后来另建的对象。也不要创建好闭包以后，只给外面的变量名 `model`/`optimizer` 重新赋值，然后以为闭包自动改绑了。

下面保持 19.5 的 loss、全局梯度范数、裁剪与更新逻辑，相关 helpers 在附录 C；本节的新重点是**将其绑定在恢复对象上**。

| 时间点 | 参数/状态含义 | 允许发生的事 |
|---|---|---|
| 刚恢复 | $\theta_7,S_7$ | 还没执行新 update |
| forward/loss/grad | 用 $\theta_7$ 算 loss 与梯度 | loss 表示更新前参数，不是 $\theta_8$ |
| 更新前检查 | loss、范数必须有限 | 失败则未执行 optimizer.update |
| 调用 update 并求值 | 更新为 $\theta_8,S_8$ | 正常期望 step 从 7 到 8 |
| 返回日志信息 | 返回仍是先前的 loss | 不会自动重新算更新后 loss |

这张表是纸面版本追踪，不是真实运行结果。更新前非有限检查不等于更新后参数已通过所有数值检查；本代码没有更新后有限值检查，也不提供 update 失败后的回滚。

In [ ]:
def make_resumed_train_step(model, optimizer, *, clip_max_norm=1.0):
    # Build a fresh gradient transform bound to THIS restored model, not an old global callable.
    loss_and_grad = nn.value_and_grad(model, loss_fn)

    def stable_step(X, Y):
        loss_before_update, grads = loss_and_grad(model, X, Y)
        grad_norm = global_grad_norm(grads)
        mx.eval(loss_before_update, grad_norm)
        if not bool(mx.all(mx.isfinite(loss_before_update)).item()):
            raise FloatingPointError("Training loss is NaN or Inf; optimizer not updated.")
        if not bool(mx.all(mx.isfinite(grad_norm)).item()):
            raise FloatingPointError("Gradient norm is NaN or Inf; optimizer not updated.")

        clipped_grads, preclip_norm, clip_scale = clip_grad_tree(grads, clip_max_norm)
        mx.eval(preclip_norm, clip_scale)
        if not bool(mx.all(mx.isfinite(preclip_norm)).item()):
            raise FloatingPointError("Pre-clipping gradient norm is NaN or Inf; optimizer not updated.")
        optimizer.update(model, clipped_grads)
        mx.eval(model.state, optimizer.state)
        return loss_before_update, grad_norm, preclip_norm, clip_scale

    return stable_step

## 8. Loop：先核对计数，再采样，再且仅再更新一次

`resume_update_loop` 的输入同时包含 model、optimizer、train_step 和 train_rng。**接口没有用身份断言证明这三者配对**；正确调用顺序必须保证 train_step 就由传入的这对对象创建。这也是末尾只给一条一致编排路径的原因。

按函数的五段读：

1. 检查 completed/target 计数，要求调用者提供训练 token 流，检查最小长度。本函数不读取 corpus 或自动建立 split。
2. 在采样前核对 optimizer step 与 supplied completed，避免拿旧 metadata 计数重启已经前进过的对象。
3. `range(completed_updates, target_total_updates)` 的循环次数是两个数的差。纸面上 7→10 应执行 3 次更新，不是 10 次；相等时不采样、不更新，返回空日志。
4. 每次先再读 step，然后用**同一个持续推进的** train_rng 采样，得到 X/Y，调用绑定的 train_step 一次。
5. 读取更新后的 step，要求它恰好是之前 +1；通过后才增加本地 completed 并追加日志。

数据需提前满足与恢复 stoi 相同的编码、正确训练 split、合法 ID/dtype 和上下文容量等条件。`get_batch` 只做有限的 shape/长度参数检查，不认证语料版本或 token 映射正确性。不能把“这份整数流能切片”理解成它与 checkpoint 配套。

### 日志的一行怎么读？

若本轮从 step 7 更新到 8，日志中 `completed_updates=8` 与 `training_loss_pre_update` 放在同一行，但这个 loss 用的是 $\theta_7$。`gradient_norm_preclip` 是裁剪前范数，`clip_scale` 说明这次缩放，而不是 loss 或学习率。完整窗口 X/Y 依旧 `(B,T)`，Y 已右移，不再 shift 第二次。

### 失败不是自动回到起点

- loss/范数检查失败发生在 update 前，但 sampler 已经抽过该 batch，train_rng 可能已经前进；不能声称所有状态未变。
- update、求值或更新后 step 检查失败时，参数/优化器可能已部分或完整改变，没有事务回滚。
- 任一异常会中止函数，不能假定已返回可靠的完整 progress_log，也不能用旧 completed 数字盲目重试。
- 本循环不自动存 checkpoint；返回日志不是新 checkpoint 文件。恢复后的第一步之前应先按 20.5 做所需核对，本节不运行它。

In [ ]:
def resume_update_loop(model, optimizer, train_step, train_tokens, *,
                       completed_updates, target_total_updates, batch_size,
                       block_size, train_rng):
    if type(completed_updates) is not int or completed_updates < 0:
        raise ValueError("completed_updates must be a non-negative integer from metadata.")
    if type(target_total_updates) is not int or target_total_updates < completed_updates:
        raise ValueError("target_total_updates must be >= completed_updates.")
    if train_tokens is None:
        raise ValueError("Supply the explicitly restored training split; this helper does not load corpus.")
    if train_tokens.shape[0] < block_size + 1:
        raise ValueError("Training split is too short for block_size+1 tokens.")

    # Check progress BEFORE sampling or updating; stale progress cannot restart a loop.
    step_path, step_value = find_named_step_leaf(optimizer.state)
    if not isinstance(step_value, mx.array):
        raise TypeError(f"Adam step at {step_path} is not an MLX array.")
    mx.eval(step_value)
    actual_step = int(step_value.item())
    if actual_step != completed_updates:
        raise ValueError(f"Adam step {actual_step} does not match supplied completed_updates {completed_updates}.")

    progress_log = []
    for _ in range(completed_updates, target_total_updates):
        before_path, before_value = find_named_step_leaf(optimizer.state)
        mx.eval(before_value)
        before_step = int(before_value.item())
        if before_step != completed_updates:
            raise ValueError(f"Progress changed unexpectedly: expected {completed_updates}, got {before_step}.")

        X, Y, _ = get_batch(train_tokens, batch_size, block_size, train_rng)
        mx.eval(X, Y)
        loss_before_update, grad_norm, preclip_norm, clip_scale = train_step(X, Y)

        after_path, after_value = find_named_step_leaf(optimizer.state)
        mx.eval(loss_before_update, after_value, model.state, optimizer.state)
        after_step = int(after_value.item())
        if after_step != before_step + 1:
            raise RuntimeError(f"Optimizer step must advance exactly once: {before_step} -> {after_step}.")
        completed_updates = after_step
        progress_log.append({
            "completed_updates": completed_updates,
            "training_loss_pre_update": float(loss_before_update.item()),
            "gradient_norm_preclip": float(grad_norm.item()),
            "gradient_norm_used_for_clipping": float(preclip_norm.item()),
            "clip_scale": float(clip_scale.item()),
        })
    return progress_log

## 9. 为什么“能接着优化”仍不等于“原轨迹复现”？

本节代码恢复模型与 optimizer，核对记录，并接上由调用者显式提供的数据和 RNG。它不会替调用者恢复 Python/MLX RNG、数据游标、随机调用次数、shuffle 顺序或 scheduler。

`train_rng` 要是同一对象持续使用的、具有 `randrange` 方法的 RNG；只按旧 seed 新建一个 RNG 通常回到起点，不代表保存时的中间状态。模型构造也可能消耗 MLX 随机性，因此真正需要精确复现时，状态恢复的时序和环境都必须单独设计，不能只把 `rng_state_saved` 设为 True。

在只掌握权重、Adam 与可靠元数据时，可将结果表述为“在明确条件下继续优化”；没有完整证据，不承诺后续每个 batch、loss 或参数逐位相同。[20.5](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/164c4bd84abcf40e72d076b54cea9027c21438c5/notebooks/91_step20_5_resume_verification.ipynb)的小输入输出/参数比较也只是那些具体检查，不证明长期质量与完整随机轨迹。

自查（不提供答案代码）：

1. 为什么 metadata_path.parent 通常不是 artifact_root？
2. 恢复后 model 正确，但 train_step 仍绑定旧 optimizer，会造成怎样的对象错配？
3. step 7→8 的日志行为什么保存更新前 loss？
4. 在非有限梯度被拒绝时，哪些状态可能已经改变？
5. 两份状态都有 step=7、且文件 hash 都匹配，为什么仍需确认保存来源？

## 可选复用附录 A · 原 MiniGPT 完整定义

下面不是新恢复算法，而是 [15.4 完整模型](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/164c4bd84abcf40e72d076b54cea9027c21438c5/notebooks/65_step15_4_implement_minigpt.ipynb)的原有定义。阅读主线可先跳过；**未来实际调用 reconstruct_model 前必须按顺序定义这些类**，不能只运行上面的调用参考。

没有外部预训练下载、没有替换模型、没有自动 forward。这里的完整类使新 work 副本不依赖旧 kernel；依赖的 imports 在本节已有。

### A1. Causal Multi-Head Attention

输入/输出 `(B,T,C)`，分头 `(B,H,T,D)`；因果 mask 仍在 forward 内创建。这里只复用，不修改参数命名或 dtype/shape 策略。

In [ ]:
import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)

### A2. MLP 与 TransformerBlock

保留带 bias 的两层 MLP、GELU 和两条 Pre-Norm RMSNorm 残差路径，输出形状仍是 `(B,T,C)`。不是为 checkpoint 新造的结构。

In [ ]:
class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.

### A3. MiniGPT 汇总

从 token/position embeddings，经独立 Blocks 到 raw logits `(B,T,V)`。类定义本身不代表已经加载训练权重；必须经主线 load_model_weights_strict。

In [ ]:
class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


# Definition only. No MiniGPT forward is executed in this master notebook.

## 可选复用附录 B · 20.2 optimizer loader

[20.2](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/164c4bd84abcf40e72d076b54cea9027c21438c5/notebooks/88_step20_2_save_optimizer_state.ipynb)已解释路径、NPZ 数组、manifest 与目标模板匹配。这里保留原完整加载函数，只定义一次，不新增另一种格式。

按空行读四段：①格式/文件；②从 manifest 建 path_to_value；③对照当前状态逐叶检查；④tree_unflatten 后赋值并求值。主线第 5 节已解释两层检查与失败边界，不要求把整段当作未讲解的新实验直接运行。

In [ ]:
def restore_optimizer_state_from_files(optimizer, arrays_path, manifest_path):
    with Path(manifest_path).open("r", encoding="utf-8") as f:
        manifest = json.load(f)
    if manifest.get("format_version") != 2:
        raise ValueError("Unsupported optimizer-state format version.")
    if manifest.get("optimizer_class") != type(optimizer).__name__:
        raise ValueError("Optimizer class does not match saved state.")

    archive = mx.load(str(arrays_path))
    path_to_value = {}
    for item in manifest["arrays"]:
        value = archive[item["key"]]
        if list(value.shape) != item["shape"] or str(value.dtype) != item["dtype"]:
            raise ValueError(f"Saved array metadata mismatch at {item['path']}")
        path_to_value[item["path"]] = value
    for item in manifest["python_scalars"]:
        value = item["value"]
        expected_type = item["python_type"]
        if expected_type == "NoneType":
            if value is not None:
                raise ValueError(f"Scalar type mismatch at {item['path']}")
        elif type(value).__name__ != expected_type:
            raise ValueError(f"Scalar type mismatch at {item['path']}")
        path_to_value[item["path"]] = value

    # Validate against the target optimizer BEFORE mutating optimizer.state.
    current_flat = tree_flatten(optimizer.state)
    current_paths = [str(path) for path, _ in current_flat]
    if set(current_paths) != set(path_to_value):
        raise ValueError("Saved optimizer-state paths do not match the current optimizer.")
    rebuilt_items = []
    for path, current_value in current_flat:
        path_text = str(path)
        saved_value = path_to_value[path_text]
        current_is_array = isinstance(current_value, mx.array)
        saved_is_array = isinstance(saved_value, mx.array)
        if current_is_array != saved_is_array:
            raise ValueError(f"Array/non-array identity mismatch at {path_text}")
        if current_is_array:
            if tuple(current_value.shape) != tuple(saved_value.shape):
                raise ValueError(f"Shape mismatch at {path_text}: {current_value.shape} vs {saved_value.shape}")
            if current_value.dtype != saved_value.dtype:
                raise ValueError(f"Dtype mismatch at {path_text}: {current_value.dtype} vs {saved_value.dtype}")
        elif type(current_value) is not type(saved_value):
            raise ValueError(f"Non-array leaf type mismatch at {path_text}")
        rebuilt_items.append((path, saved_value))

    # Only assign after every path/type/shape/dtype check has passed.
    rebuilt = tree_unflatten(rebuilt_items)
    optimizer.state = rebuilt
    mx.eval(optimizer.state)

# Definition only. No state is loaded during material preparation.

## 可选复用附录 C · Loss、梯度范数与 clipping

沿用 [19.5](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/164c4bd84abcf40e72d076b54cea9027c21438c5/notebooks/86_step19_5_baseline_training_run.ipynb)对应的训练稳定性主题与原函数逻辑。`loss_fn` 将 `(B,T,V)` 展平成 `(B*T,V)`，Y 展平成 `(B*T,)`，逐 token 交叉熵后取平均。原始函数名、参数、形状和运算都保留。

In [ ]:
def loss_fn(model, X, Y):
    # X/Y are (B,T); Y is already shifted by the sampler.
    logits = model(X)  # (B,T,V)
    B, T, V = logits.shape
    flat_logits = mx.reshape(logits, (B * T, V))
    flat_targets = mx.reshape(Y, (B * T,))
    per_token = nn.losses.cross_entropy(flat_logits, flat_targets, axis=-1, reduction="none")
    return mx.mean(per_token)  # scalar MLX array

### C2. 裁剪只复用既有实现

global_grad_norm 汇总梯度树的平方和再开方；clip_grad_tree 用一个全局比例缩放所有叶子，并返回原范数和 scale。它不是逐个参数各自 clip，不在本节重新设计优化算法。

In [ ]:
def global_grad_norm(grad_tree):
    leaves = tree_flatten(grad_tree)
    squared_sums = [mx.sum(mx.square(leaf)) for _, leaf in leaves]
    if not squared_sums:
        raise ValueError("Gradient tree has no array leaves.")
    return mx.sqrt(sum(squared_sums))

def clip_grad_tree(grad_tree, max_norm, eps_clip=1e-12):
    if max_norm <= 0:
        raise ValueError("clip_max_norm must be positive.")
    norm = global_grad_norm(grad_tree)
    mx.eval(norm)
    scale = mx.minimum(
        mx.array(1.0, dtype=mx.float32),
        mx.array(max_norm, dtype=mx.float32) / (norm + eps_clip),
    )
    return tree_map(lambda grad: grad * scale, grad_tree), norm, scale

## 可选复用附录 D · 同一个 train_rng 驱动的既有 sampler

`get_batch` 从一维 token 流取每段 T+1 个 ID，`[:-1]` 作 X、`[1:]` 作 Y，堆叠得到 `(B,T)`。`randrange(0,n-block_size)` 的上界不包含自身，最后合法起点为 `n-block_size-1`，保证那一段仍有 T+1 项。

返回第三项 starts 供记录/理解；当前主循环用 `_` 丢弃它，因此没有保存 sampler 游标或全部抽样历史。无需重新运行旧 Notebook，但未来须显式提供真实同映射 token 流与合适 RNG。

In [ ]:
def get_batch(data, batch_size, block_size, rng):
    if len(data.shape) != 1:
        raise ValueError("Expected a 1-D token stream.")
    if type(batch_size) is not int or batch_size < 1 or type(block_size) is not int or block_size < 1:
        raise ValueError("batch_size and block_size must be positive integers.")
    n = data.shape[0]
    if n < block_size + 1:
        raise ValueError("Split must contain at least block_size + 1 tokens.")
    starts = [rng.randrange(0, n - block_size) for _ in range(batch_size)]
    xs, ys = [], []
    for start in starts:
        chunk = data[start:start + block_size + 1]
        xs.append(chunk[:-1])
        ys.append(chunk[1:])
    return mx.stack(xs, axis=0), mx.stack(ys, axis=0), starts

## 10. 唯一的未来调用参考：先定义全部依赖，再使用同一组对象

下面全部是注释，没有执行任何 IO、构造、采样或更新。它不是“去掉所有 # 就一定能跑”的零输入脚本：路径、ADAM_CONFIG、同映射训练 split、真实 RNG/数据状态与目标步数必须事先确定；全部 helpers 与附录类必须已定义。缺一项就停止，不用假值补齐。

同一个 metadata 产生 artifacts 与模型配置；同一对 model/optimizer 先恢复、再核对、再创建 train_step；同一对对象和该 train_step 最后进入 loop。不要在这些阶段之间重新构造对象、重新 init optimizer 或替换变量。返回的日志不会自动写回 metadata 或保存新 checkpoint。

In [ ]:
# Future work-copy orchestration; select both locations explicitly.
# metadata_path = Path(".../run_metadata/<run-id>/run_metadata.json")
# artifact_root = Path(".../the-original-artifact-root-used-by-20.3")
# metadata_path, metadata, stoi = load_checkpoint_metadata(metadata_path)
# artifacts = resolve_checkpoint_artifacts(artifact_root, metadata)
# model, optimizer = create_model_and_optimizer(metadata)
# load_model_weights_strict(model, artifacts["weights"])
# restore_optimizer_state_from_files(optimizer, artifacts["optimizer_arrays"], artifacts["optimizer_manifest"])
# verify_completed_update_count(optimizer, metadata)
# train_step = make_resumed_train_step(model, optimizer)
# progress_log = resume_update_loop(
#     model, optimizer, train_step, explicit_train_tokens,
#     completed_updates=metadata["training_progress"]["completed_updates"],
#     target_total_updates=desired_total_updates,
#     batch_size=chosen_batch_size, block_size=metadata["architecture"]["max_seq_len"],
#     train_rng=explicit_restored_rng,
# )

## 参考与本次修订边界

- [20.1 权重保存](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/164c4bd84abcf40e72d076b54cea9027c21438c5/notebooks/87_step20_1_save_model_weights.ipynb)：独立权重产物。
- [20.2 状态序列化](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/164c4bd84abcf40e72d076b54cea9027c21438c5/notebooks/88_step20_2_save_optimizer_state.ipynb)：目标结构匹配与替换边界。
- [20.3 元数据](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/164c4bd84abcf40e72d076b54cea9027c21438c5/notebooks/89_step20_3_save_configuration_metadata.ipynb)：规范映射、两种位置、hash/seed/progress 的证据范围。

本次只修订 20.4 的教学组织、移除未使用的旧同名 loader、合并重复注释调用参考；最终保留函数的签名与函数体不变。旧完整模型与训练辅助代码移到复用附录，不增算法、不添加可执行调用。

历史修正计数不归零，旧验收不冒充本次已验收；本次等待独立静态验收。所有 code cells 未执行、无输出，不运行模型、训练、Notebook、真实文件读取或恢复测试；静态通过不是运行通过。本轮停在 20.4，不修改或推进其他课节。